# DATASCI 503, Group Work 5: Cross Validation

**Instructions:** During lab section, and afterward as necessary, you will collaborate in two-person teams (assigned by the GSI) to complete the problems that are interspersed below. The GSI will help individual teams encountering difficulty, make announcements addressing common issues, and help ensure progress for all teams. During lab, feel free to flag down your GSI to ask questions at any point!

## Getting Started and Data imports

In this assignment, we will explore logistic regression and $k$-fold cross validation on the NHANES dataset. We begin by loading and preparing the data, then build up an understanding of cross-validation and implement it from scratch. Let's start by importing the relevant packages.

In [ ]:
import inspect

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn.metrics
from sklearn.base import clone
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

import os

if os.path.isdir("fixtures"):
    !cp -n fixtures/*.

### **Problems**

---

**Problem 1: Dataset Setup**

Our favorite (and only) dataset we have used in the group work assignments is back! Please take the appropriate time to load in the `NHANES` datasets and merge them into a single DataFrame called `df`.

In [ ]:
# your code here

In [ ]:
# Test assertions
assert "df" in dir(), "The variable 'df' should be defined"
assert hasattr(df, "shape"), "df should be a DataFrame"
assert df.shape[0] > 5000, "The merged dataframe should have more than 5000 rows"
print("All tests passed!")

---

**Problem 2: Variable Setup and Selection**

For this problem, we will again try to predict individuals that have high-density lipoprotein (HDL) cholesterol of greater than 60. An HDL of 60 **mg/dL** or higher is often viewed as protective against heart disease—this is typically the level you'd like to aim for, if possible.

For this task please do the following:

1. Create the binary indicator variable called `HDL>60`. Also, use the following features for predictive purposes: Gender, Age, Weight, Height, BMI, WaistSize, HouseholdSize, and Ethnicity. You may need to refer to the docs to figure out their variable names:
   - [HDL_L](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/HDL_L.htm)
   - [DEMO_L](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.htm)
   - [BMX_L](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.htm)

2. Rename the variable as their English-legible version from the previous point (e.g., `RIAGENDR` becomes `Gender`, `RIDAGEYR` becomes `Age`, `BMXWT` becomes `Weight`, and so on).

3. Drop all missing values.

Store the resulting DataFrame in a variable called `my_df`.

In [ ]:
# your code here


In [ ]:
# Test assertions
assert "my_df" in dir(), "The variable 'my_df' should be defined"
assert my_df.shape[0] / df.shape[0] > 0.8, "At least 80% of rows should remain after dropping NaN"
assert "HDL>60" in my_df.columns, "my_df should have an 'HDL>60' column"
print("All tests passed!")

---

**Problem 3: Training and Testing Split**

Please split your data into a train and test set, with 70% of observations in the train set and 30% in the test set.

* Use the `train_test_split` function from sklearn.
* Use `random_state=42`.
* Stratify the sampling to include roughly the same distribution of response values in each set. [Why should we stratify?](https://scikit-learn.org/stable/modules/cross_validation.html#stratified-k-fold)

Store the results in `X_train`, `X_test`, `y_train`, and `y_test`.

In [ ]:
# your code here

In [ ]:
# Test assertions
assert "X_train" in dir(), "X_train should be defined"
assert "X_test" in dir(), "X_test should be defined"
assert "y_train" in dir(), "y_train should be defined"
assert "y_test" in dir(), "y_test should be defined"
print("All tests passed!"

---

## Cross-Validation

What is the point of CV?

In this part of the assignment, you'll learn about K-fold cross validation, implement it from scratch, and then apply it to several classification models on the NHANES dataset. Additionally, you will be asked to write answers to explain WHY we do the things we do. Please feel free to ask instructors for guidance if cross-validation is still new to you.

In $k$-fold cross-validation, we partition a dataset into $k$ equally sized non-overlapping subsets $S$. For each subset $S_i$, a model is trained on $S \setminus S_i$ and evaluated on $S_i$. The cross-validation estimator of the prediction's error is the average of the prediction errors obtained on each fold.

Let's start with a simple example that will help us understand how CV works. We'll create a synthetic binary classification dataset.

In [ ]:
# Generate a synthetic binary classification dataset
X_synth, y_synth = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)

## K-Fold CV by Hand

We'll look at two ways to do cross-validation. First we'll do it "by hand." Then we'll show how sklearn makes it easy.

First, we divide the data into 10 parts (also known as folds).

In [ ]:
n_splits = 10  # 10-fold
rng = np.random.default_rng(0)  # make a random number generator with fixed seed
permutation = rng.permutation(len(X_synth))  # create a shuffling of the indices of our data
splits = np.split(permutation, n_splits)  # make folds

In [ ]:
# e.g., these are the *indices* of the datapoints in "fold 3"
splits[3]

In [ ]:
# and these are the y values for the corresponding samples
y_synth[splits[3]]

Now, we'll assess predictive performance 10 times. Each time we'll hold out a different fold and use the rest of the data for training.

In [ ]:
# we'll look at two metrics of predictive performance,
# and store the results in these arrays
missclass = np.zeros(len(splits))
aurocs = np.zeros(len(splits))  # since this is binary classification, we can assess via auroc

# iterate through folds
for i in range(len(splits)):
    # create test/train split for this held-out fold
    folds_list = list(splits)  # copy list
    test = folds_list.pop(i)  # pop out the ith fold
    training = np.concatenate(folds_list)  # combine all remaining data for training

    # fit model
    model = LogisticRegression()  # make estimator
    model.fit(X_synth[training], y_synth[training])  # fit estimator using the training data

    # assess metric for predictive performance using test data
    missclass[i] = np.mean(model.predict(X_synth[test]) != y_synth[test])  # get misclassification on test data
    aurocs[i] = sklearn.metrics.roc_auc_score(
        y_synth[test], model.predict_proba(X_synth[test])[:, 1]
    )  # get auroc on test data

In [ ]:
missclass  # misclassification rate for each of the 10 test/train splits

In [ ]:
aurocs  # auroc for each of the 10 test/train splits

In [ ]:
# report conclusions: mean(scores) +/- sqrt(tau^2/K)
print(
    f"Mean misclassification rate: {missclass.mean():.4f}, "
    f"plus minus {np.sqrt(missclass.var() / n_splits):.4f}"
)
print(f"Auroc: {aurocs.mean():.4f}, plus minus {np.sqrt(aurocs.var() / n_splits):.4f}")

`sklearn.base.clone`

When performing cross-validation, you need a **fresh copy** of the model for each fold — otherwise the model retains its fitted state from a previous fold. The helper `sklearn.base.clone` does exactly this: it returns a new estimator with the same hyperparameters but no fitted state.

```python
from sklearn.base import clone

original = LogisticRegression(C=0.5, max_iter=1000)
fresh_copy = clone(original)  # same hyperparameters, unfitted
```

### **Problems**

---

**Problem 4: Implementing K-Fold CV**

Write a function called `KFoldCV` that performs K-fold cross-validation **from scratch**. You must implement the fold-splitting and training loop yourself.

The function takes 4 arguments:

1. `X`: The predictors array.
2. `y`: The response variable array.
3. `model`: An sklearn model object on which we can call `fit` and `predict`.
4. `K`: An integer representing the number of folds (default: 10).

The function should return a numpy array of classification accuracies for each fold.

**Requirements:**
- Do **not** use `sklearn.model_selection.KFold`, `cross_val_score`, or `cross_validate`.
- Use a fixed random seed so that results are reproducible (the starter code provides `rng`).
- Use `clone(model)` to obtain a fresh estimator for each fold.

The by-hand example above is a good reference for the logic you need.

In [ ]:
def KFoldCV(X: np.ndarray | pd.DataFrame, y: np.ndarray | pd.Series, model, K: int = 10) -> np.ndarray:
    """
    Perform K-fold cross-validation from scratch.

    Parameters:
        X: Feature matrix (DataFrame or array)
        y: Labels (Series or array)
        model: An sklearn estimator with fit() and predict()
        K: Number of folds (default 10)

    Returns:
        numpy array of length K with the classification accuracy for each fold
    """
    X = np.asarray(X)
    y = np.asarray(y)
    n = len(X)
    rng = np.random.default_rng(42)

    # your code here


In [ ]:
# Test assertions
_test_model = LogisticRegression(max_iter=500)
_test_scores = KFoldCV(X_train, y_train, _test_model, K=5)
assert len(_test_scores) == 5, "Should return 5 scores for K=5"
assert all(0 <= s <= 1 for s in _test_scores), "Each score should be between 0 and 1"
assert _test_scores.mean() > 0.5, "Mean accuracy should be better than random guessing"
print("All tests passed!")

## K-Fold CV Using sklearn

Now that you've implemented cross-validation from scratch, let's see how sklearn makes this much easier. This is for your reference — in the remaining problems you will continue using your own `KFoldCV` function.

In [ ]:
model = LogisticRegression()

Set up the $k$-fold cross-validation configuration. For example, using 10 folds:

* `n_splits=10` means 10-fold
* `shuffle=True` meaning random folds
* `random_state=42` means we'll get the same random folds each time

In [ ]:
cv = KFold(n_splits=10, random_state=42, shuffle=True)

With a single call to `cross_val_score`, we can evaluate the model using cross-validation. Here, we'll use accuracy as the performance metric, but you can choose other metrics like precision, recall, etc.

In [ ]:
# Calculate accuracies across folds
scores = cross_val_score(model, X_synth, y_synth, scoring="accuracy", cv=cv, n_jobs=-1)
scores

array([0.87, 0.86, 0.83, 0.89, 0.88, 0.86, 0.85, 0.87, 0.9 , 0.86])

In [ ]:
print(f"Mean accuracy: {scores.mean():.4f}, plus minus {np.sqrt(scores.var() / len(scores)):.4f}")
print(
    f"Mean misclassification rate: {1 - scores.mean():.4f}, "
    f"plus minus {np.sqrt(scores.var() / len(scores)):.4f}"
)

Mean accuracy: 0.8670, plus minus 0.0060
Mean misclassification rate: 0.1330, plus minus 0.0060


If you want to use multiple metrics, use `sklearn.model_selection.cross_validate` instead.

In [ ]:
# info for each split
results = pd.DataFrame(
    sklearn.model_selection.cross_validate(
        model,
        X_synth,
        y_synth,
        cv=cv,
        scoring=("accuracy", "roc_auc"),
        return_train_score=True,
    )
)
results

,fit_time,score_time,test_accuracy,train_accuracy,test_roc_auc,train_roc_auc
0,0.004868,0.003870,0.87,0.874444,0.910879,0.937207
1,0.003247,0.003478,0.86,0.880000,0.926250,0.936828
2,0.009184,0.003953,0.83,0.881111,0.900641,0.939317
3,0.003914,0.008320,0.89,0.872222,0.942222,0.934140
4,0.010125,0.011236,0.88,0.877778,0.951600,0.933526
5,0.003367,0.003644,0.86,0.876667,0.939382,0.934303
6,0.003236,0.003451,0.85,0.883333,0.910364,0.937373
7,0.003378,0.003510,0.87,0.880000,0.890405,0.939992
8,0.003286,0.004082,0.90,0.870000,0.961851,0.932240
9,0.003322,0.003988,0.86,0.876667,0.958788,0.932327


Note the results also include train performance (because we set `return_train_score=True`). It can sometimes be interesting to see discrepancy between train and test performance. Usually train performance is a bit better. If train performance is *a lot* better, your estimator may have "too much flexibility" (though sometimes you may also be experiencing so-called "benign overfitting" in which case your estimator is actually just fine...).

### **Problems**

---

**Problem 5a: Execution**

Run your `KFoldCV` function using Logistic Regression with $K=10$.

Store the result in a variable called `cv_scores` and make sure the output is visible in the notebook.

**Note:** If you get a warning that the model has not reached convergence, add the argument `max_iter=1000` to the `LogisticRegression` instance.

In [ ]:
# your code here


In [ ]:
# Test assertions
assert "cv_scores" in dir(), "cv_scores should be defined"
assert len(cv_scores) == 10, "Should have 10 fold scores"
print("All tests passed!")

---

**Problem 5b: Visualizing Generalization**

One motivation for cross-validation is to assess the generalizability of your model on unseen data. One can use CV to estimate the distribution of the population risk of our model.

Create a **bar plot** of the 10 per-fold accuracies from `cv_scores`. Add a **horizontal dashed line** at the mean accuracy. Label the x-axis `"Fold"`, the y-axis `"Accuracy"`, and give the plot the title `"Per-Fold CV Accuracies"`.

Store the figure in `fig_cv` and the Axes object in `ax_cv`.

In [ ]:
# your code here

In [ ]:
# Test assertions
assert "fig_cv" in dir(), "fig_cv should be defined"
assert "ax_cv" in dir(), "ax_cv should be defined"
_bars = ax_cv.patches
assert len(_bars) == 10, "Should have 10 bars (one per fold)"
print("All tests passed!")

## Bias-Variance Tradeoff

It is very helpful to think about the bias-variance tradeoff in cross-validation. In CV, the number of folds to use (the value of $k$) is an important decision. Imagine repeating the learning procedure on multiple datasets. The lower the value for $k$, the higher the bias in the error estimates and the less variance **across datasets**. Conversely, when $k$ is set equal to the training+val sample size, the error estimate is then very low in bias but has the possibility of high variance **across datasets**.

Why? Some intuitions (but not mathematically rigorous proof) here:

While there is no overlap between the test sets on which the models are evaluated, there is overlap between the training sets for all $k>2$. The overlap is largest for leave-one-out cross-validation. This means that the learned models are correlated, i.e., dependent, and the variance of the sum of correlated variables increases with the amount of covariance:

$$
\text{Var}\left(\sum_i X_i\right) = \sum_i \sum_j \text{Cov}(X_i, X_j)
$$

Therefore, leave-one-out cross-validation has large variance in comparison to CV with smaller $k$. To summarize, larger $k$ means less bias towards overestimating the true expected error (as training folds will be closer to the total dataset) but higher variance and higher running time (as you are getting closer to the limit case: Leave-One-Out CV).

For more fun facts and simulation about bias-variance tradeoff and cross validation, please see [this post](https://stats.stackexchange.com/questions/61783/bias-and-variance-in-leave-one-out-vs-k-fold-cross-validation/357749#357749).

### **Problems**

---

**Problem 6a: Working with Regularization**

It turns out that if you look at the LogisticRegression module in sklearn, it uses an L2 penalty by default! You can see for yourself on the [documentation page for sklearn.linear_model.LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

One problem: we just accepted the default regularization term `C=1.0`. How do we know this was the right choice? Let's find out.

Write a function called `KFoldCV_L2` that performs K-Fold validation with different values of the regularization parameter $C$. The values to consider should be $10^{-5}, 10^{-4}, \ldots, 10^{4}$ (i.e., 10 values total).

The function should take as input `X`, `y`, and `K` (default 10) and return a dictionary with C values as keys and mean validation accuracy as values.

**Hint:** Use your `KFoldCV` function from Problem 4.

In [ ]:
def KFoldCV_L2(X: np.ndarray | pd.DataFrame, y: np.ndarray | pd.Series, K: int = 10) -> dict:
    """
    Perform K-fold CV for logistic regression with different L2 regularization strengths.

    Parameters:
        X: pandas DataFrame of features
        y: pandas Series of labels
        K: number of folds (default 10)

    Returns:
        dict: C values as keys, mean validation accuracy as values
    """
    # your code here

In [ ]:
# Test assertions
l2_results = KFoldCV_L2(X_train, y_train)
assert isinstance(l2_results, dict), "Should return a dictionary"
assert len(l2_results) == 10, "Should have results for 10 C values"
print("All tests passed!")

In [ ]:
# Display the results
l2_results

---

**Problem 6b: Visualizing Regularization**

Create a **line plot with markers** showing the mean CV accuracy (y-axis) against `log10(C)` (x-axis) for the values in `l2_results`. Label the x-axis `"log10(C)"`, the y-axis `"Mean CV Accuracy"`, and title the plot `"Accuracy vs Regularization Strength"`.

Store the figure in `fig_l2`, the Axes in `ax_l2`, and your chosen best value of C in a variable called `best_C`.

In [ ]:
# your code here

In [ ]:
# Test assertions
assert "fig_l2" in dir(), "fig_l2 should be defined"
assert "ax_l2" in dir(), "ax_l2 should be defined"
assert "best_C" in dir(), "best_C should be defined"
_lines = ax_l2.get_lines()
assert len(_lines) >= 1, "Should have at least one line in the plot"
print("All tests passed!")

---

**Problem 7a: Train and Test Evaluation**

Please now retrain your final model using the best regularization value (`best_C`) you found in Problem 6b, trained on **all** training data. Then, evaluate your model's performance on the test set.

Store the trained model in `final_model` and the test accuracy in `test_accuracy`.

In [ ]:
# your code here

In [ ]:
# Test assertions
assert "final_model" in dir(), "final_model should be defined"
assert "test_accuracy" in dir(), "test_accuracy should be defined"
assert 0 <= test_accuracy <= 1, "test_accuracy should be between 0 and 1"
print("All tests passed!")

---

**Problem 7b: Retraining Justification**

Why do we retrain the model on the **full** training set (all data except the test set) after using cross-validation to select hyperparameters?

- **A.** To verify that the model doesn't overfit the training data
- **B.** To maximize the amount of training data used for the final model, since CV was only needed for hyperparameter selection
- **C.** To ensure that the test set was not accidentally used during cross-validation
- **D.** To reset the random seed before final evaluation

Set your answer in the variable `answer_7b` (e.g., `answer_7b = "A"`).

In [ ]:
# your code here

In [ ]:
# Test assertions
assert "answer_7b" in dir(), "answer_7b should be defined"
assert answer_7b in ("A", "B", "C", "D"), "answer_7b should be one of 'A', 'B', 'C', 'D'"
print("All tests passed!")

---

**Problem 7c: Performance Evaluation**

First, compute the **baseline accuracy** — the accuracy you would get by always predicting the majority class. Store it in `baseline_accuracy`.

Then, given the baseline accuracy and the logistic regression test accuracy, which of the following best characterizes the model's performance?

- **A.** The model is performing very well — it substantially outperforms the baseline
- **B.** The model is performing modestly — it only marginally outperforms the majority-class baseline
- **C.** The model is performing poorly — it is worse than the baseline
- **D.** The model is performing well — accuracy above 70% is always considered strong

Set your answer in the variable `answer_7c` (e.g., `answer_7c = "A"`).

In [ ]:
# your code here

In [ ]:
# Test assertions
assert "baseline_accuracy" in dir(), "baseline_accuracy should be defined"
assert "answer_7c" in dir(), "answer_7c should be defined"
assert answer_7c in ("A", "B", "C", "D"), "answer_7c should be one of 'A', 'B', 'C', 'D'"
print("All tests passed!")

## Compare with Another Estimator

Let's see whether a different model can do better. We'll try K-Nearest Neighbors (KNN). First, a quick demo on the synthetic data using the same CV folds:

In [ ]:
model2 = KNeighborsClassifier(5)
scores2 = cross_val_score(
    model2, X_synth, y_synth, scoring="accuracy", cv=cv, n_jobs=-1
)  # note, using same folds, cv

print(
    f"Mean Accuracy: {scores2.mean():.4f}, plus minus {np.sqrt(scores2.var() / len(scores2)):.4f}"
)

It seems that 5-NN is worse than logistic regression on the synthetic data, by a margin that is well in excess of the spread. Let's try KNN on our NHANES data.

### **Problems**

---

**Problem 8: K-Folds on K-Nearest Neighbors**

Using your `KFoldCV` function you created in Problem 4, please run CV on 2-NN classification (K-Nearest Neighbors with `n_neighbors=2`).

Store the result in a variable called `knn_cv_scores`.

In [ ]:
# Test assertions
assert "knn_cv_scores" in dir(), "knn_cv_scores should be defined"
assert len(knn_cv_scores) == 10, "Should have 10 fold scores"
print("All tests passed!")

---

**Problem 9: Finding the Right K for KNN**

Similar to what you did for regularized logistic regression, please run a 10-fold CV that assesses model performance for 10 different values of `n_neighbors`: [1, 3, 5, 10, 15, 20, 25, 35, 50, 100]. You should evaluate each fold with each `n_neighbors` value.

Write a function called `KFoldCV_NN` that takes as input `X`, `y`, and `K` (default 10) and returns a dictionary. The dictionary should have the number of neighbors considered as keys and the mean validation accuracy as values.

**Hint:** Use your `KFoldCV` function from Problem 4.

In [ ]:
def KFoldCV_NN(X, y, K=10):  # noqa: N802, N803
    """
    Perform K-fold CV for KNN with different numbers of neighbors.

    Parameters:
        X: pandas DataFrame of features
        y: pandas Series of labels
        K: number of folds (default 10)

    Returns:
        dict: n_neighbors values as keys, mean validation accuracy as values
    """
    # your code here

In [ ]:
# Test assertions
nn_results = KFoldCV_NN(X_train, y_train)
assert isinstance(nn_results, dict), "Should return a dictionary"
assert len(nn_results) == 10, "Should have results for 10 n_neighbors values"
print("All tests passed!")

In [ ]:
# Display the results
nn_results

---

**Problem 10: Retraining KNN on Training Data**

Retrain your KNN model on the full training set. Make sure you use the best number of neighbors as determined from the previous part.

Store the best number of neighbors in `BEST_K` and the trained model in `knn_final_model`.

In [ ]:
# your code here

In [ ]:
# Test assertions
assert "BEST_K" in dir(), "BEST_K should be defined"
assert "knn_final_model" in dir(), "knn_final_model should be defined"
assert BEST_K in [1, 3, 5, 10, 15, 20, 25, 35, 50, 100], "BEST_K should be one of the tested values"
print("All tests passed!")

---

**Problem 11: Test Set Evaluation for KNN**

Evaluate your final KNN model's performance on the test set.

Store the test accuracy in `knn_test_accuracy`.

In [ ]:
# your code here

In [ ]:
# Test assertions
assert "knn_test_accuracy" in dir(), "knn_test_accuracy should be defined"
assert 0 <= knn_test_accuracy <= 1, "knn_test_accuracy should be between 0 and 1"
print("All tests passed!")